# 📗 검색 품질 지표와 RAG 진단: Context Precision·Recall·Relevancy

교안 01에서는 답변을 평가했습니다. 그런데 답변 점수만으로는 원인을 알 수 없습니다.

- 필요한 문서를 못 가져와서 틀렸을 수도 있고(검색 문제), 문서는 제대로 가져왔는데 모델이 문서에 없는 내용을 덧붙였을 수도 있습니다(답변 생성 문제).
- 둘을 구분하려면 검색 결과의 품질을 따로 평가해야 합니다.

**이 교안의 흐름**
- 검색 품질 지표 3개를 계산합니다.
- 교안 01의 생성 지표 3개를 더해 지표 6개로 종합 평가하고, 합격선과 기준선 비교로 고장 위치를 진단합니다.
- 실무에서 쓸 지표를 고르는 기준을 정리합니다.

자료는 고용노동부 「일·생활 균형을 위한 유연근무 활용 매뉴얼」(2024)에서 발췌한 44개 절입니다.

**이 단원에서 배우는 지표 6종**

RAGAS 지표는 RAG가 만든 결과를 0~1 점수로 평가하며, 높을수록 좋습니다. 지표 이름의 Context(컨텍스트)는 검색 문서를 가리킵니다.

| 교안 | 구분 | 지표 | 무엇을 평가하나 | 점수가 낮으면 |
|---|---|---|---|---|
| 01 | 생성 품질 | Faithfulness(답변 충실도) | **근거**: 답변이 검색 문서에 있는 내용만 말했는가 | 답변이 문서에 없거나 문서와 다른 내용을 말함 |
| 01 | 생성 품질 | Answer Relevancy(답변 관련성) | **주제**: 답변이 질문에 맞는 이야기인가(사실 여부는 보지 않고, 정답 답변 없이 판정) | 질문과 다른 이야기를 하거나 확답을 피함 |
| 01 | 생성 품질 | Answer Correctness(답변 정확도) | **사실**: 답변의 사실이 정답 답변과 일치하는가 | 정답 답변과 다른 사실을 말하거나 빠뜨림 |
| 02 | 검색 품질 | Context Precision(컨텍스트 정밀도) | **순서**: 정답 답변을 만드는 데 쓸모 있는 문서가 검색 순위 위쪽에 있는가 | 쓸모 있는 문서가 아래 순위로 밀림 |
| 02 | 검색 품질 | Context Recall(컨텍스트 재현율) | **빠짐**: 정답 답변에 필요한 내용을 검색 문서가 빠짐없이 담았는가 | 필요한 문서가 빠짐 |
| 02 | 검색 품질 | Context Relevancy(컨텍스트 관련성) | **엉뚱함**: 가져온 검색 문서가 질문과 관련 있는가(정답 답변 없이 판정) | 질문과 관련 없는 문서를 가져옴 |

지표는 평가 샘플 한 건의 네 칸 가운데 일부를 비교해 점수를 냅니다. ragas는 네 칸을 아래 영어 필드 이름으로 받습니다.

| 필드 | 이 교안의 용어 | 뜻 |
|---|---|---|
| `user_input` | 질문 | 사용자가 한 질문 |
| `retrieved_contexts` | 검색 문서 | 검색기가 질문으로 찾아 답변 모델에 넘긴 문서 조각 목록 |
| `response` | 답변 | RAG가 만든 답변 |
| `reference` | 정답 답변 | 사람이 근거 원문을 확인하고 미리 써 둔 모범 답안 문장 |

질문과 정답 답변을 짝지어 모은 것을 실무에서는 **골든셋**(정답셋)이라고 부릅니다. "골든셋이 있다"는 평가할 질문마다 정답 답변이 준비됐다는 뜻입니다.

<img src="images/02_rag_metric_map.png" width="1050" alt="지표 6종이 평가 샘플의 네 칸(질문 user_input, 검색 문서 retrieved_contexts, 답변 response, 정답 답변 reference) 가운데 어느 칸끼리 비교하는지 원으로 표시한 표. 생성 품질: Faithfulness(답변 충실도, 근거)는 검색 문서·답변, Answer Relevancy(답변 관련성, 주제)는 질문·답변, Answer Correctness(답변 정확도, 사실)는 답변·정답 답변. 검색 품질: Context Precision(컨텍스트 정밀도, 순서)과 Context Recall(컨텍스트 재현율, 빠짐)은 검색 문서·정답 답변, Context Relevancy(컨텍스트 관련성, 엉뚱함)는 질문·검색 문서. 정답 답변 칸에 원이 있는 지표는 골든셋이 있어야 계산한다.">

**검색 지표도 정답 문서가 아니라 정답 답변을 씁니다.** 정답 문서(`relevant_doc_ids`)는 질문의 답이 들어 있다고 사람이 정한 문서이고, 앞 단원의 Precision@K·Recall@K가 이 문서 ID로 검색 결과를 셉니다. RAGAS의 Context Precision·Context Recall은 문서 ID를 세지 않고, 평가용 LLM이 검색 문서를 읽어 정답 답변에 필요한 내용이 들어 있는지 판정합니다.

**이름은 비슷하지만 다른 것을 보는 두 쌍**
- **Answer Relevancy와 Answer Correctness:** "10시간을 넘어야 연장근로수당이 나옵니다"(정답은 8시간)는 질문에 맞는 이야기라 Answer Relevancy는 높지만, 사실이 틀려 Answer Correctness는 낮습니다.
- **Context Recall과 Context Relevancy:** 질문에 필요한 문서가 둘인데 1위 문서 하나만 가져오면, 그 문서는 질문과 관련 있어 Context Relevancy는 높지만 나머지 하나가 빠져 Context Recall은 낮습니다.

교안 01에서 생성 품질 지표 3개를 계산했습니다. 이 교안은 검색 품질 지표 3개를 계산하고, 교안 01의 생성 지표와 함께 지표 6개로 RAG를 종합 평가해 고장 위치를 진단합니다.

<img src="images/10_lesson02_goals.png" width="1050" alt="이번 교안에서 배울 것: 검색 품질 평가와 고장 위치 진단. 유연근무 매뉴얼 RAG의 기준선과 고장 설정 세 개로 실습한다. 1 Context Precision(컨텍스트 정밀도): 쓸모 있는 문서가 위에 있는가, 2 Context Recall(컨텍스트 재현율): 필요한 정보를 다 가져왔는가, 3 Context Relevancy(컨텍스트 관련성): 가져온 문서가 질문과 관련 있는가, 4 종합 평가와 고장 진단: 지표 6개로 판정, 합격선과 기준선 비교.">

**학습 목표** (그림의 번호와 같습니다)

1. **Context Precision으로 쓸모 있는 문서가 검색 순위 위쪽에 있는지 평가합니다.** 문서마다 받은 쓸모 판정(1 또는 0)과 순위로 점수가 계산되는 방식을 이해합니다.
2. **Context Recall로 정답 답변에 필요한 정보를 검색 문서가 모두 담았는지 평가합니다.** 문서 수를 줄인 설정과 순서만 바꾼 설정에서 Context Precision과 어떻게 다르게 움직이는지 비교합니다.
3. **Context Relevancy로 정답 답변 없이 검색 문서가 질문과 관련 있는지 평가합니다.** 0·1·2점 판정 두 번으로 점수가 정해지는 방식을 이해합니다.
4. **교안 01의 생성 지표까지 지표 6개로 RAG를 종합 평가하고 고장 위치를 진단합니다.** 합격선(절대 판정)으로 써도 되는지, 기준선 비교(상대 판정)로 어디가 고장 났는지 판정하고, 골든셋이 있는지에 따라 쓸 지표를 고릅니다.

## 1. 검색 품질을 문서 내용으로 평가하기

### 1-1. 검색 지표 세 개의 계산 방식

세 검색 지표는 질문(`user_input`), 검색 문서(`retrieved_contexts`), 정답 답변(`reference`) 가운데 필요한 칸만 쓰고 답변(`response`)은 쓰지 않습니다. 지표마다 평가용 LLM이 판정하는 대상과 점수 계산 방식이 다릅니다.

| 지표 | 평가용 LLM이 판정하는 것 | 판정 횟수(질문 하나) | 점수 계산 | 순위 반영 | 정답 답변 |
|---|---|---|---|---|---|
| Context Precision | 문서 하나가 정답 답변을 만드는 데 쓸모 있는가(1 또는 0) | 검색 문서 수만큼 | 쓸모 있는 문서가 나온 순위마다 1위부터 그 순위까지 쓸모 있는 문서의 비율을 구해 평균 | 반영 | 필요 |
| Context Recall | 정답 답변의 주장 하나가 검색 문서로 뒷받침되는가(1 또는 0) | 1번(주장 나누기와 판정을 한 번에, 문서는 이어 붙임) | 뒷받침되는 주장 수 ÷ 정답 답변의 전체 주장 수 | 반영 안 함 | 필요 |
| Context Relevancy | 검색 문서 묶음이 질문과 관련 있는가(0·1·2점) | 2번(표현이 다른 지시문 2개) | 두 점수를 각각 2로 나눠 평균 | 반영 안 함 | 필요 없음 |

- **이름:** 커리큘럼 표기는 Context Relevancy이고, ragas의 클래스 이름은 `ContextRelevance`입니다.


### 1-2. 앞 단원의 검색 평가와 다른 점

앞 단원의 Precision@K·Recall@K는 사람이 정한 정답 문서 ID가 검색 결과에 있는지를 셌습니다. RAGAS 검색 지표는 정답 문서 ID 목록이 없어도 되고, Context Precision·Context Recall은 대신 정답 답변이 있어야 합니다.

| | 앞 단원 Precision@K·Recall@K | RAGAS Context Precision·Context Recall |
|---|---|---|
| 필요한 것 | 정답 문서 ID | 정답 답변 |
| 판정 | 문서 ID가 검색 결과에 있는지 셉니다 | 평가용 LLM이 문서 본문을 읽고 판정합니다 |
| 비용·재현성 | 빠르고 매번 같은 값이 나옵니다 | LLM 호출 비용이 들고 실행마다 조금씩 달라집니다 |
| 강한 상황 | 임베딩 모델·k처럼 검색기 설정을 여러 번 바꿔 비교할 때 | 청크를 다시 나눠 문서 ID가 바뀌었을 때, 같은 내용이 다른 문서에도 있을 때 |

**그래서 실무에서는 둘을 함께 씁니다.**

- **검색기 튜닝:** 임베딩 모델·청크 크기·k를 고르는 반복 실험은 문서 ID 기반 지표(Recall@K 등)로 빠르고 싸게 비교합니다. 검색기만 따로 평가할 때 가장 흔히 쓰는 방식입니다.
- **RAG 전체와 운영 점검:** 답변까지 한 번에 보거나 정답 문서 목록이 없을 때는 RAGAS 검색 지표를 씁니다. 운영 중 질문처럼 정답 답변도 없으면 Context Relevancy만 쓸 수 있습니다.
- **골든셋 만들기:** 정답 답변은 근거 원문을 확인하며 쓰므로, 그때 연 문서의 ID를 함께 적어 두는 데 드는 수고는 거의 없습니다. 질문·정답 답변·정답 문서 ID를 한 번에 만들어 두면 두 방식을 모두 쓸 수 있어 이득입니다. 이 교안의 평가 질문도 이렇게 만들었습니다.
- **주의:** 정답 문서 ID는 청크 나누기를 바꾸면 다시 매겨야 합니다. 정답 답변은 문서를 어떻게 나누든 그대로 쓸 수 있습니다.

## 2. 실습 준비

### 2-1. 경로·모델·평가용 LLM

경로·환경 변수, 답변 모델, 질문 임베딩 모델을 준비하고, 다음 셀에서 교안 01과 같은 설정으로 평가용 LLM(`evaluator_llm`)을 만듭니다. 6절에서 교안 01의 Answer Relevancy·Answer Correctness도 계산하므로 평가용 임베딩 모델(`evaluator_embeddings`)도 함께 만듭니다.

In [ ]:
import os
import json
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"

# API 키 읽기: 단원 폴더 .env를 먼저 읽고, 없는 키만 저장소 최상위 .env에서 채웁니다
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")

model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")  # 답변 모델. .env에 없으면 기본 모델
llm = ChatOpenAI(model=model_name, use_responses_api=True)

# 검색 질문 임베딩: 배포한 문서 벡터와 같은 모델·차원이어야 합니다
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large",
    dimensions=768,
    check_embedding_ctx_length=False,  # 입력 길이를 미리 검사해 나누는 자동 처리를 끕니다
)
chroma_dir = data_dir / "본실습" / "chroma"

print("답변 모델:", model_name)

In [ ]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory

os.environ["RAGAS_DO_NOT_TRACK"] = "true"  # ragas가 익명 사용 통계를 외부 서버로 보내지 않게 합니다

# 평가용 LLM: ragas 0.4 지표는 LangChain 모델 대신 OpenAI SDK의 비동기 클라이언트를 받습니다
# ragas 0.4.3은 gpt-5.6-luna처럼 버전에 소수점이 든 이름을 추론 모델로 알아보지 못해 max_tokens를 그대로 보내고, API가 400 오류로 거절합니다
eval_model_name = os.getenv("OPENAI_EVAL_MODEL", "gpt-6-luna")
eval_client = AsyncOpenAI()

KOREAN_RULE = "모든 출력 문장(질문·주장·이유)은 한국어로 작성하세요."

evaluator_llm = llm_factory(
    eval_model_name,
    client=eval_client,
    max_tokens=8192,            # 응답 길이 상한. 추론 모델은 생각하는 데 쓴 토큰도 여기에 들어가므로 기본값(1024)보다 크게 둡니다
    system_prompt=KOREAN_RULE,  # 이 평가용 LLM을 쓰는 모든 지표에 한국어 규칙이 적용됩니다
)

# gpt-5.6-luna처럼 소수점 버전 모델을 쓰려면 아래 세 줄의 주석을 풀어 추론 모델용 인자로 직접 바꿉니다(ragas 저장소 이슈 #2573)
# evaluator_llm.model_args["max_completion_tokens"] = evaluator_llm.model_args.pop("max_tokens")
# evaluator_llm.model_args["temperature"] = 1.0
# evaluator_llm.model_args.pop("top_p")

# Answer Relevancy·Answer Correctness가 문장 사이 의미 유사도를 계산할 때 쓰는 임베딩 모델
evaluator_embeddings = embedding_factory(
    "openai",
    model="text-embedding-3-large",  # ragas 0.4.3 임베딩은 차원 인자를 받지 않아 기본 3072차원을 씁니다
    client=eval_client,
)

print("평가용 LLM:", eval_model_name)

### 2-2. 배포 Chroma 열기

`data/본실습/chroma/`에는 매뉴얼 44개 절의 본문과 벡터가 미리 저장되어 있습니다. 문서를 다시 임베딩하지 않고, 검색할 때 질문만 임베딩합니다.

In [ ]:
from langchain_chroma import Chroma


def evidence_id(doc):
    """검색 문서의 문서 ID(메타데이터 source_id)를 반환합니다."""
    return doc.metadata["source_id"]


vector_store = Chroma(
    collection_name="flexwork_manual",
    persist_directory=str(chroma_dir),
    embedding_function=embedding_model,
    create_collection_if_not_exists=False,  # 배포한 컬렉션이 없으면 새로 만들지 않고 오류를 냅니다
)

# 문서 ID로 본문을 찾는 dict: 저장된 본문을 읽기만 하고 다시 임베딩하지 않습니다
stored_documents = vector_store.get(include=["documents", "metadatas"])
documents_by_id = {}
for text, meta in zip(stored_documents["documents"], stored_documents["metadatas"]):
    documents_by_id[meta["source_id"]] = text

print("매뉴얼 문서:", len(documents_by_id))

In [ ]:
# 상위 4개 문서를 찾는 검색기입니다. 뒤에서 만들 네 가지 설정이 모두 이 4개 안에서 문서를 고릅니다.
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

### 2-3. 평가 질문과 정답 답변

평가 질문 6개는 모두 정답 문서가 둘 이상입니다. 그래서 검색 문서 수를 줄인 설정에서는 정답 답변에 필요한 내용이 빠질 수 있습니다.

- **정답 답변(reference):** 질문마다 2~3문장입니다. 질문 6개와 정답 답변의 짝이 이 교안의 골든셋입니다.
- **정답 문서(relevant_doc_ids):** ragas 지표는 이 목록을 쓰지 않습니다. 이 교안에서는 질문과 동떨어진 문서로 비교할 때와 원문을 확인할 때만 씁니다.

In [ ]:
# 평가 질문 6개 읽기
questions = json.loads((data_dir / "본실습" / "questions.json").read_text(encoding="utf-8"))

# 질문 ID로 정답 문서 목록을 찾는 dict
relevant_ids = {entry["question_id"]: entry["relevant_doc_ids"] for entry in questions}

# 질문마다 한 행씩 표로 보기
question_rows = []
for entry in questions:
    question_rows.append({
        "질문 ID": entry["question_id"],
        "유형": entry["question_type"],
        "질문": entry["question"],
        "정답 답변": entry["reference"],
        "정답 문서": ", ".join(entry["relevant_doc_ids"]),
    })
display(pd.DataFrame(question_rows))

### 2-4. 매뉴얼 RAG와 네 가지 설정

같은 RAG로 비교의 기준이 되는 설정인 **기준선**(baseline) 하나와, 고장을 하나씩 넣은 설정 셋을 만들어 어느 지표의 점수가 떨어지는지 봅니다. 고장은 일부러 넣었지만 답변은 답변 모델이 실제로 만들고, 점수도 평가용 LLM이 실제로 매깁니다.

- **기준선:** 잘 동작한다고 확인한 설정입니다. 개선안을 시험할 때는 지금 운영 중인 설정이, 무언가를 바꾼 뒤 고장 위치를 찾을 때는 바꾸기 전 설정이 기준선이 됩니다. 이 교안은 고장 위치를 찾으므로 바꾸기 전 설정을 기준선으로 둡니다.

<img src="images/15_four_settings.png" width="1050" alt="진단에 쓰는 설정 네 개: 기준선 하나, 고장 하나씩 셋. 기준선은 1위부터 4위 문서와 기본 프롬프트, 순위 흐트림은 같은 문서를 4위부터 1위 순서로 넘기고 기본 프롬프트, 검색 누락은 1위 문서 하나만 넘기고 기본 프롬프트, 근거 이탈은 1위부터 4위 문서와 일반 지식과 수치를 덧붙이라는 프롬프트. 순위 흐트림과 검색 누락은 검색 단계 고장, 근거 이탈은 답변 생성 단계 고장. 질문마다 상위 4개를 한 번 검색하고 네 설정이 함께 씁니다.">

문서를 문자열로 묶는 함수, 프롬프트, 체인은 앞 단원과 같은 방식으로 만듭니다. 기본 프롬프트(`answer_prompt`)와 근거 이탈 프롬프트(`loose_prompt`)를 한 셀에 둡니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate


def format_documents(docs):
    """검색 문서의 문서 ID·제목·본문을 하나의 문자열로 묶습니다."""
    # 문서마다 "[문서 ID] 제목" 줄을 앞에 붙여, 모델이 어느 문서의 내용인지 구분하게 합니다
    return "\n\n".join([f"[{evidence_id(doc)}] {doc.metadata['title']}\n{doc.page_content}" for doc in docs])


# 기본 프롬프트: 문서에 있는 내용만 근거로 답하게 합니다
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 회사 인사팀의 유연근무 상담 챗봇입니다. 아래 문서에 있는 내용만 근거로 2~4문장으로 답하세요. "
               "문서에 없는 내용은 추측하지 말고 문서에서 확인할 수 없다고 말하세요."),
    ("human", "문서:\n{context}\n\n질문: {question}"),
])
answer_chain = answer_prompt | llm

# 근거 이탈 프롬프트: 같은 문서를 주면서 일반 인사·노무 지식과 구체 수치를 덧붙이라고 지시합니다(답변 생성 단계의 고장)
loose_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 회사 인사팀의 유연근무 상담 챗봇입니다. 아래 문서를 참고하되, 인사·노무 실무 경험과 근로기준법 등 일반 법률 지식을 "
               "적극적으로 덧붙여 2~4문장으로 풍부하게 답하세요. 관련 법 조항 번호, 금액, 기간 같은 구체적인 수치를 함께 알려 주면 좋습니다."),
    ("human", "문서:\n{context}\n\n질문: {question}"),
])
loose_chain = loose_prompt | llm

In [ ]:
CONFIGS = ["기준선", "순위 흐트림", "검색 누락", "근거 이탈"]  # 답변 생성과 진단 출력이 이 순서로 설정을 반복합니다


def pick_documents(config, top4):
    """설정에 맞게 상위 4개 검색 문서에서 답변에 넘길 문서를 골라 반환합니다."""
    if config == "순위 흐트림":
        # 기준선과 같은 문서 4개를 순서만 뒤집어 넘깁니다
        return list(reversed(top4))
    elif config == "검색 누락":
        # 1위 문서 하나만 넘깁니다. 정답 문서가 둘 이상인 질문에서 정보가 빠집니다
        return top4[:1]
    else:
        # 기준선과 근거 이탈은 같은 상위 4개를 씁니다(근거 이탈은 프롬프트만 다름)
        return top4

이 교안에서는 설정 하나와 질문 하나의 조합을 **실행**이라고 부릅니다. 실행 하나는 평가 샘플의 네 칸(`user_input`, `retrieved_contexts`, `response`, `reference`)과 설정 이름·검색 문서 ID를 담습니다.

아래 셀은 질문마다 상위 4개를 한 번 검색하고 네 설정으로 각각 답변을 만듭니다. 기본 프롬프트를 쓰는 세 설정은 `batch`로 한 번에 보내 동시에 답변을 받고, 프롬프트가 다른 근거 이탈만 따로 보냅니다. 질문 임베딩 6번과 답변 생성 24번을 호출합니다.

In [ ]:
# 동시에 보낼 요청 수의 상한입니다. 한꺼번에 너무 많이 보내면 API 요청 한도(분당 요청·토큰 수)를 넘어 429 오류가 날 수 있습니다.
# 32는 ragas evaluate()의 기본 동시 실행 수(RunConfig의 max_workers, 16)보다 넉넉하게 잡은 값입니다. 429 오류가 나면 줄이고, 키 한도가 넉넉하면 늘립니다.
MAX_CONCURRENCY = 32

runs = []
for entry in questions:
    # 질문마다 한 번만 검색하고, 네 설정이 이 상위 4개에서 문서를 고릅니다
    top4 = retriever.invoke(entry["question"])

    # 네 설정의 검색 문서와 답변 입력을 먼저 모두 만듭니다
    docs_by_config = {}
    inputs_by_config = {}
    for config in CONFIGS:
        docs_by_config[config] = pick_documents(config, top4)
        inputs_by_config[config] = {"question": entry["question"], "context": format_documents(docs_by_config[config])}

    # 기본 프롬프트를 쓰는 세 설정은 batch로 한 번에 보내 동시에 답변을 받습니다 (결과는 입력 순서대로 돌아옵니다)
    normal_configs = ["기준선", "순위 흐트림", "검색 누락"]
    normal_answers = answer_chain.batch(
        [inputs_by_config[config] for config in normal_configs],
        config={"max_concurrency": MAX_CONCURRENCY},  # max_concurrency: 동시에 보낼 요청 수 상한(기본은 제한 없음에 가까운 스레드 풀 크기). 평가와 같은 상한을 씁니다
    )
    responses = {}
    for config, answer in zip(normal_configs, normal_answers):
        responses[config] = answer.text.strip()

    # 근거 이탈만 프롬프트가 달라 근거 이탈 체인으로 따로 보냅니다
    responses["근거 이탈"] = loose_chain.invoke(inputs_by_config["근거 이탈"]).text.strip()

    # ragas 지표에 그대로 넘길 수 있게 평가 샘플의 칸 이름(user_input 등)으로 저장합니다
    for config in CONFIGS:
        docs = docs_by_config[config]
        runs.append({
            "config": config,
            "question_id": entry["question_id"],
            "user_input": entry["question"],
            "retrieved_contexts": [doc.page_content for doc in docs],
            "retrieved_ids": [evidence_id(doc) for doc in docs],
            "response": responses[config],
            "reference": entry["reference"],
        })
        print("생성:", entry["question_id"], config, "| 검색 문서 수:", len(docs), "| 답변 앞부분:", responses[config][:50])
print("실행 수:", len(runs), "| 설정 수:", len(CONFIGS), "| 질문 수:", len(questions))

In [ ]:
# 실행 24개를 한 표로 보기 (답변은 앞 60자만)
preview_rows = []
for run in runs:
    preview_rows.append({
        "설정": run["config"],
        "질문 ID": run["question_id"],
        "검색 문서 ID": ", ".join(run["retrieved_ids"]),
        "답변 앞 60자": run["response"][:60],
    })
display(pd.DataFrame(preview_rows))

점수를 계산하기 전에 검색 문서 ID 열로 설정이 의도대로 만들어졌는지 확인합니다. 순위 흐트림은 기준선과 같은 ID가 거꾸로 놓이고 검색 누락은 ID가 하나뿐이어야, 뒤의 점수 차이를 그 고장 때문이라고 해석할 수 있습니다.

## 3. Context Precision(컨텍스트 정밀도): 쓸모 있는 문서가 위에 있는가

### 3-1. 평균 정밀도

**Context Precision**은 쓸모 있는 문서가 검색 순위 위쪽에 모여 있는지 평가합니다.

- **판정 목록 만들기:** 평가용 LLM이 검색 문서마다 "정답 답변을 만드는 데 쓸모 있는가"를 1 또는 0으로 판정하고, 판정을 순위대로 늘어놓습니다.
- **순위별 정밀도:** 판정이 1인 순위에서만 "1위부터 그 순위까지 나온 쓸모 있는 문서 수 ÷ 그 순위"를 구합니다. 판정이 0인 순위는 건너뜁니다.
- **평균 정밀도(average precision):** 순위별 정밀도를 모두 더해 쓸모 있는 문서 수로 나눕니다. 쓸모 있는 문서가 위에 있을수록 나누는 순위가 작아 점수가 높습니다.

<img src="images/11_context_precision_rank.png" width="1050" alt="Context Precision(컨텍스트 정밀도): 쓸모 있는 문서가 위에 있는가. 질문 '신청 반려와 승인 후 보안 문제에 따른 해제는 각각 어떤 절차를 마련하나요?', 정답 답변 '반려 시 이의제기 절차, 보안 문제 시 관리자의 해제 요청 절차'. 순서 A는 1위 신청과 승인 절차(1), 2위 승인 후 업무관리(1), 3위 재택근무의 보안대책(0), 4위 재택근무 신청·해지 절차의 설계(0)로 판정 [1, 1, 0, 0], (1/1 + 2/2) ÷ 2 = 1.0. 순서 B는 1위 재택근무의 보안대책(0), 2위 신청과 승인 절차(1), 3위 승인 후 업무관리(1), 4위 재택근무 신청·해지 절차의 설계(0)로 판정 [0, 1, 1, 0], (1/2 + 2/3) ÷ 2 = 0.58. 같은 문서 4개, 순서만 다름.">

### 3-2. 설정별 Context Precision

`runs`의 실행마다 Context Precision을 계산해 `precision_rows`에 한 행씩 쌓습니다. 검색 문서 하나마다 평가용 LLM을 한 번 호출하므로 질문 하나에 13번(4 + 4 + 1 + 4, 검색 누락만 문서 1개), 질문 6개면 78번 호출합니다.

평가는 교안 01 4-2처럼 실행 24개의 `ascore`를 먼저 모두 만들어 두고, `run_limited`로 동시 실행 수를 제한하면서 `tqdm_asyncio.gather`로 한꺼번에 실행합니다.

In [ ]:
import asyncio

from ragas.metrics.collections import ContextPrecision
from tqdm.asyncio import tqdm_asyncio

# gather는 넘긴 작업을 한꺼번에 시작하므로, 세마포어로 동시에 실행되는 평가 작업 수를 MAX_CONCURRENCY개로 묶습니다
semaphore = asyncio.Semaphore(MAX_CONCURRENCY)  # 동시에 실행 중인 작업이 MAX_CONCURRENCY개가 되면 나머지는 자리가 날 때까지 기다립니다


async def run_limited(task):
    """세마포어 자리가 날 때까지 기다렸다가 작업 하나를 실행하고 결과를 돌려줍니다."""
    async with semaphore:
        return await task


context_precision = ContextPrecision(llm=evaluator_llm)  # 검색 문서마다 정답 답변을 만드는 데 쓸모 있는지(1 또는 0) 판정하는 지표

# 실행 24개의 평가를 기다리지 않고 모두 시작해 둡니다
tasks = []
for run in runs:
    tasks.append(context_precision.ascore(
        user_input=run["user_input"],
        reference=run["reference"],
        retrieved_contexts=run["retrieved_contexts"],  # 순위가 점수에 들어가므로 검색된 순서 그대로 넘깁니다
    ))

# 동시에 계산하고 모두 끝나면 결과를 runs 순서대로 받습니다 (진행 막대로 끝난 개수를 보여 줍니다)
results = await tqdm_asyncio.gather(*[run_limited(task) for task in tasks], desc="Context Precision")

# 실행마다 점수 한 행을 만듭니다. 뒤의 지표도 같은 네 열을 씁니다
precision_rows = []
for run, result in zip(runs, results):
    precision_rows.append({
        "설정": run["config"],
        "질문 ID": run["question_id"],
        "지표": "Context Precision",
        "점수": result.value,
    })
    print("평가:", run["config"], run["question_id"], "| 점수:", round(result.value, 3))

In [ ]:
# 행은 설정, 열은 질문 ID인 표 (칸마다 점수가 하나라 pivot을 씁니다)
display(pd.DataFrame(precision_rows).pivot(index="설정", columns="질문 ID", values="점수").round(2))

표는 기준선 행과 비교해 읽습니다.

- **순위 흐트림:** 기준선과 같은 문서를 순서만 뒤집은 설정이므로, 기준선보다 낮다면 그 차이는 문서 순서에서 생긴 것입니다.
- **검색 누락:** 1위 문서 하나만 판정하므로 점수가 높아도 빠진 문서가 없다는 뜻은 아닙니다. 빠진 문서는 Context Recall로 확인합니다.

### 3-3. 정답 답변 없이 쓰는 변형

정답 답변이 없을 때는 `ContextPrecisionWithoutReference`를 씁니다. 계산 방법은 같고, 문서가 쓸모 있는지를 정답 답변 대신 RAG가 만든 답변(`response`)에 비추어 판정합니다.

- **약점:** 답변이 검색 문서를 벗어나 엉뚱하게 작성되면, 질문에 필요한 좋은 문서도 쓸모없다고 판정될 수 있습니다.
- **읽는 법:** 그래서 이 변형의 점수는 Faithfulness가 높은 답변에서만 검색 품질로 읽습니다. Faithfulness가 낮은 답변에서 점수가 낮으면 검색보다 답변을 먼저 의심합니다.

## 4. Context Recall(컨텍스트 재현율): 정답 답변에 필요한 정보를 다 가져왔는가

### 4-1. 정답 답변의 주장 나누기, 검색 문서 대조, 비율 계산

**Context Recall**은 정답 답변의 주장 가운데 검색 문서로 뒷받침되는 주장의 비율입니다. 교안 01의 Faithfulness와 계산 방식이 같고, 나누는 대상이 답변이 아니라 정답 답변입니다.

- **입력:** 질문(`user_input`)·검색 문서(`retrieved_contexts`)·정답 답변(`reference`). 답변은 쓰지 않습니다.
- **계산:** 평가용 LLM이 정답 답변을 주장으로 나누고, 주장마다 검색 문서로 뒷받침되는지 1 또는 0으로 판정합니다. 나누기와 판정을 한 번의 호출로 합니다.
- **공식:** Context Recall = 검색 문서로 뒷받침되는 주장 수 ÷ 정답 답변의 전체 주장 수
- **순위 반영 안 함:** 검색 문서를 이어 붙여 한 번에 읽으므로, 문서의 순위는 점수에 영향을 주지 않습니다.

<img src="images/12_context_recall.png" width="1050" alt="Context Recall(컨텍스트 재현율): 필요한 정보를 다 가져왔는가. 정답 답변의 두 주장 ① 반려 시 이의제기 절차를 마련한다, ② 보안 문제가 생기면 해제를 요청하는 절차를 마련할 수 있다. 검색 문서는 신청과 승인 절차 하나이고 승인 후 업무관리는 검색되지 않음. ①은 검색 문서로 뒷받침됨, ②는 뒷받침 안 됨. 뒷받침된 주장 1개 ÷ 정답 답변 주장 2개 = 0.5.">

q33의 정답 답변으로 계산 과정을 따라가 봅니다.

1. **주장으로 나누기:** 평가용 LLM이 정답 답변을 "반려 시 이의제기 절차를 마련한다"와 "보안 문제가 생기면 해제를 요청하는 절차를 마련할 수 있다"의 두 주장으로 나눕니다.
2. **검색 문서와 대조:** 검색 문서가 「신청과 승인 절차」 하나뿐이고 「승인 후 업무관리」가 검색되지 않았다면, 첫 주장은 1, 둘째 주장은 0입니다.
3. **비율 계산:** 주장 2개 중 1개만 뒷받침되므로 1 ÷ 2 = 0.5입니다. 두 문서가 모두 검색됐다면 어느 순위에 있든 2 ÷ 2 = 1.0입니다.
4. 주장을 몇 개로 나눌지는 평가용 LLM이 정하므로 실제 주장 수는 달라질 수 있습니다.

Context Recall이 1보다 낮으면 검색 문서에서 찾지 못한 정답 내용이 있다는 뜻입니다. 이때는 검색 문서 수(k)를 늘리거나 검색 방식을 바꿔 빠진 정보를 채웁니다.

평가용 LLM이 정답 답변에서 주장을 하나도 뽑지 못하면 비율을 계산할 수 없어 NaN(값 없음)이 됩니다.

**생각해 보기**: 정답 문서 두 개 중 하나만 검색됐는데 Context Recall이 1.0일 수 있을까요?

<details><summary>정답 보기</summary>

네, 그럴 수 있습니다. Context Recall은 문서 ID가 아니라 정답 답변의 주장마다 검색 문서 본문으로 뒷받침되는지를 봅니다. 빠진 문서의 내용이 검색된 다른 문서에도 적혀 있으면 정답 답변의 주장이 모두 뒷받침되어 1.0이 됩니다.

</details>

### 4-2. 설정별 Context Recall

Context Precision과 같은 방식으로 `recall_rows`를 만듭니다. 실행마다 평가용 LLM을 한 번씩, 모두 24번 호출하며, 앞 절처럼 한꺼번에 실행합니다.

In [ ]:
from ragas.metrics.collections import ContextRecall

context_recall = ContextRecall(llm=evaluator_llm)  # 정답 답변의 주장마다 검색 문서로 뒷받침되는지(1 또는 0) 판정하는 지표

# 실행 24개의 평가를 기다리지 않고 모두 시작해 둡니다
tasks = []
for run in runs:
    tasks.append(context_recall.ascore(
        user_input=run["user_input"],
        retrieved_contexts=run["retrieved_contexts"],  # 문서를 하나씩 묻지 않고 이어 붙여 한 번에 판정받습니다
        reference=run["reference"],
    ))

# 동시에 계산하고 모두 끝나면 결과를 runs 순서대로 받습니다 (진행 막대로 끝난 개수를 보여 줍니다)
results = await tqdm_asyncio.gather(*[run_limited(task) for task in tasks], desc="Context Recall")

# Context Precision과 같은 네 열로 실행마다 한 행을 만듭니다
recall_rows = []
for run, result in zip(runs, results):
    recall_rows.append({
        "설정": run["config"],
        "질문 ID": run["question_id"],
        "지표": "Context Recall",
        "점수": result.value,
    })
    print("평가:", run["config"], run["question_id"], "| 점수:", round(result.value, 3))

In [ ]:
# Context Precision 표와 같이 행은 설정, 열은 질문 ID인 표
display(pd.DataFrame(recall_rows).pivot(index="설정", columns="질문 ID", values="점수").round(2))

- **검색 누락:** Context Recall이 기준선보다 내려갔다면, 1위 문서 밖에 있던 정답 내용이 검색 문서에서 빠진 것입니다.
- **순위 흐트림:** 기준선과 같은 문서를 넘기므로 Context Recall도 기준선과 비슷해야 합니다. 차이가 크면 검색이 아니라 평가용 LLM의 판정이 흔들린 것이므로 그 질문의 원문을 열어 확인합니다.

**확인 퀴즈**: 상위 4개 문서의 쓸모 판정이 [1, 1, 0, 0]인 검색이 있습니다. 5·6위에 있던 정답 문서 두 개까지 넘기자 판정이 [1, 1, 0, 0, 1, 1]이 됐습니다. Context Recall과 Context Precision은 각각 어떻게 될까요?

<details><summary>정답 보기</summary>

Context Recall은 오르거나 그대로이고, Context Precision은 1.0보다 내려갑니다.

- **Context Recall:** 빠졌던 내용이 새로 들어오면 오르고, 이미 다른 문서에 있던 내용이면 그대로입니다.
- **Context Precision:** 새로 들어온 쓸모 있는 문서 두 개가 쓸모없는 문서 두 개 아래에 놓여 내려갑니다. 정보를 더 가져왔는데도 검색이 나빠진 것처럼 보이므로 두 지표를 함께 읽습니다.

</details>

## 5. Context Relevancy(컨텍스트 관련성): 가져온 문서가 질문과 관련 있는가

### 5-1. 0·1·2점을 두 번 매겨 평균

**Context Relevancy**는 정답 답변 없이 질문과 검색 문서만으로 관련성을 평가합니다.

- **점수 매기기:** 평가용 LLM이 검색 문서 전체를 한 덩어리로 읽고, 질문과 관련 있는 정도를 0점(관련 정보 없음)·1점(일부만 관련)·2점(관련 정보 있음)으로 매깁니다.
- **두 번 매기기:** 표현이 다른 지시문으로 한 번 더 매깁니다.
- **점수 계산:** 두 점수를 각각 2로 나눠 0~1 범위로 바꾸고 평균합니다.

<img src="images/14_context_relevance.png" width="1050" alt="Context Relevancy(컨텍스트 관련성): 가져온 문서가 질문과 관련 있는가. 질문 '외부 거래처와 협조가 필요한 재택근무 팀의 근무 방식과 협업 수단은?'과 검색 문서 4개를 하나로 이어 붙여 한 번에 넘긴다. 첫 번째 지시문은 2점, 표현이 다른 두 번째 지시문은 1점. 0점 관련 정보 없음, 1점 일부만 관련, 2점 관련 정보 있음. 2점 ÷ 2 = 1.0, 1점 ÷ 2 = 0.5, 평균 (1.0 + 0.5) ÷ 2 = 0.75.">

q34(외부 거래처와 협조가 필요한 재택근무 팀)로 계산 과정을 따라가 봅니다. 점수는 설명을 위해 가정한 값입니다.

1. 검색 문서 4개를 하나로 이어 붙여 질문과 함께 평가용 LLM에 넘깁니다. 문서마다 따로 묻지 않습니다.
2. 첫 번째 지시문으로 2점을 받았다고 합시다.
3. 표현이 다른 두 번째 지시문으로는 1점을 받았다고 합시다.
4. 2점 ÷ 2 = 1.0과 1점 ÷ 2 = 0.5를 평균한 (1.0 + 0.5) ÷ 2 = 0.75가 최종 점수입니다.

- **점수 단계:** 판정마다 0·1·2 세 단계뿐이라 최종 점수도 0, 0.25, 0.5, 0.75, 1 가운데 하나입니다. 작은 품질 차이는 점수에 나타나지 않습니다.
- **장점:** 정답 답변이 필요 없어, 정답 답변을 만들어 두지 않은 질문에도 검색 품질을 평가할 수 있습니다.

**NaN이 나올 때**
- 두 판정 중 하나만 실패하면 남은 판정 하나로 점수를 냅니다. 둘 다 실패하면 오류를 내지 않고 NaN을 돌려줍니다.
- 키나 모델 이름이 잘못돼 생긴 오류도 몇 번 다시 시도한 뒤 NaN으로 바뀝니다. 점수표의 NaN은 점수가 아니라 평가에 실패한 칸입니다.
- 일부 칸만 NaN이면 평가 셀을 다시 실행하고, 모든 칸이 NaN이면 다시 실행하기 전에 키와 평가용 LLM 설정부터 확인합니다.

### 5-2. 설정별 Context Relevancy

Context Precision과 같은 방식으로 `relevance_rows`를 만듭니다. 실행마다 평가용 LLM을 두 번씩, 모두 48번 호출하며, 앞 절처럼 한꺼번에 실행합니다.

In [ ]:
from ragas.metrics.collections import ContextRelevance

context_relevance = ContextRelevance(llm=evaluator_llm)  # 질문과 검색 문서 묶음의 관련성을 0·1·2점으로 두 번 매겨 평균하는 지표

# 실행 24개의 평가를 기다리지 않고 모두 시작해 둡니다
tasks = []
for run in runs:
    tasks.append(context_relevance.ascore(
        user_input=run["user_input"],
        retrieved_contexts=run["retrieved_contexts"],  # 정답 답변은 넘기지 않습니다
    ))

# 동시에 계산하고 모두 끝나면 결과를 runs 순서대로 받습니다 (진행 막대로 끝난 개수를 보여 줍니다)
results = await tqdm_asyncio.gather(*[run_limited(task) for task in tasks], desc="Context Relevancy")

# Context Precision과 같은 네 열로 실행마다 한 행을 만듭니다
relevance_rows = []
for run, result in zip(runs, results):
    relevance_rows.append({
        "설정": run["config"],
        "질문 ID": run["question_id"],
        "지표": "Context Relevancy",
        "점수": result.value,
    })
    print("평가:", run["config"], run["question_id"], "| 점수:", round(result.value, 3))

In [ ]:
# Context Precision 표와 같이 행은 설정, 열은 질문 ID인 표 (NaN 칸은 두 판정이 모두 실패한 자리)
display(pd.DataFrame(relevance_rows).pivot(index="설정", columns="질문 ID", values="점수").round(2))

검색 누락 행이 기준선과 비슷하다면, 넘긴 1위 문서가 질문과 관련 있어서 정보가 빠진 것이 이 지표에는 드러나지 않은 것입니다. 빠진 정보는 Context Recall 표에서 확인합니다.

### 5-3. 질문과 동떨어진 문서로 비교하기

Context Relevancy가 어떤 검색을 낮게 평가하는지 보려고, q7(유연근무 도입목적과 공감대) 질문에 q7의 정답 문서와 a1(선택근무 연차휴가수당)의 정답 문서를 각각 붙여 평가합니다. 평가용 LLM을 4번(문서 묶음 2개 × 지시문 2개) 호출합니다.

In [ ]:
# q7 질문에 붙일 두 문서 묶음: q7의 정답 문서와 a1의 정답 문서
q7_question = questions[0]["question"]  # questions의 첫 질문이 q7입니다
own_contexts = [documents_by_id[doc_id] for doc_id in relevant_ids["q7"]]
other_contexts = [documents_by_id[doc_id] for doc_id in relevant_ids["a1"]]

# 두 문서 묶음의 Context Relevancy 계산
own_result = await context_relevance.ascore(user_input=q7_question, retrieved_contexts=own_contexts)
other_result = await context_relevance.ascore(user_input=q7_question, retrieved_contexts=other_contexts)
print("질문:", q7_question)
print("q7 정답 문서 점수:", own_result.value, "| a1 정답 문서 점수:", other_result.value)

a1 정답 문서의 점수가 q7 정답 문서보다 뚜렷이 낮다면, Context Relevancy가 질문과 주제가 다른 문서 묶음을 골라낸 것입니다. 실제 RAG에서는 이 점수가 낮은 검색 결과부터 원문을 열어, 질문과 다른 주제의 문서를 가져왔는지 확인합니다.

**확인 퀴즈**: 정답 답변이 없는 새 질문 100개의 검색 품질을 먼저 훑어보려 합니다. 이 교안의 세 검색 지표 중 바로 쓸 수 있는 것은 무엇일까요? 그 점수가 높게 나와도 아직 알 수 없는 것은 무엇일까요?

<details><summary>정답 보기</summary>

Context Relevancy입니다. Context Precision과 Context Recall은 정답 답변이 필요합니다. Context Precision에는 답변을 기준으로 삼는 변형이 있지만, 답변이 검색 문서를 벗어나면 판정도 틀릴 수 있습니다.

Context Relevancy가 높아도 답변에 필요한 정보를 빠짐없이 가져왔는지는 아직 알 수 없습니다. 이것은 정답 답변을 만든 뒤 Context Recall로 확인합니다.

</details>

## 6. RAG 종합 평가와 고장 위치 진단

검색 지표는 답변을 읽지 않으므로, 답이 맞는지는 알려 주지 못합니다. 그래서 교안 01의 생성 품질 지표 세 개를 더해 지표 6개로 평가하고, 결과를 두 질문으로 나눠 읽습니다.

- **이 설정을 써도 되나?** 지표마다 정한 합격선(절대 판정)과 비교합니다.
- **어디가 고장 났나?** 기준선(baseline)과 비교해 떨어진 지표로 고장 단계를 좁힙니다.

검색과 답변 생성을 따로 평가해 고장 위치를 좁히는 방식은 Ragas 논문, DeepEval, TruLens, Azure AI Foundry의 RAG 평가가 함께 쓰는 틀입니다.

### 6-1. 교안 01의 생성 지표 세 개 계산

같은 실행 24개에 교안 01의 지표 세 개를 계산합니다. 검색 지표처럼 지표마다 한꺼번에 실행합니다.

- **Faithfulness:** 답변이 받은 검색 문서를 벗어났는지(답변 생성 단계의 고장)
- **Answer Relevancy:** 답변이 질문에서 엇나갔는지
- **Answer Correctness:** 최종 답이 정답 답변과 맞는지

In [ ]:
from ragas.metrics.collections import AnswerCorrectness, AnswerRelevancy, Faithfulness

# 교안 01의 생성 품질 지표 세 개
faithfulness = Faithfulness(llm=evaluator_llm)              # 답변의 주장 가운데 검색 문서로 뒷받침되는 비율
answer_relevancy = AnswerRelevancy(
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,                        # 거꾸로 만든 질문과 원래 질문의 유사도를 계산할 임베딩 모델
)
answer_correctness = AnswerCorrectness(
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,                        # 답변과 정답 답변의 의미 유사도를 계산할 임베딩 모델
)

# 실행 24개마다 세 지표의 평가를 기다리지 않고 모두 시작해 둡니다
faithfulness_tasks = []
relevancy_tasks = []
correctness_tasks = []
for run in runs:
    faithfulness_tasks.append(faithfulness.ascore(user_input=run["user_input"], response=run["response"], retrieved_contexts=run["retrieved_contexts"]))
    relevancy_tasks.append(answer_relevancy.ascore(user_input=run["user_input"], response=run["response"]))
    correctness_tasks.append(answer_correctness.ascore(user_input=run["user_input"], response=run["response"], reference=run["reference"]))

# 지표마다 동시에 계산하고 결과를 runs 순서대로 받습니다
faithfulness_results = await tqdm_asyncio.gather(*[run_limited(task) for task in faithfulness_tasks], desc="Faithfulness")
relevancy_results = await tqdm_asyncio.gather(*[run_limited(task) for task in relevancy_tasks], desc="Answer Relevancy")
correctness_results = await tqdm_asyncio.gather(*[run_limited(task) for task in correctness_tasks], desc="Answer Correctness")

# 검색 지표와 같은 네 열로 실행마다 지표 세 개의 행을 만듭니다
generation_rows = []
for run, f_result, r_result, c_result in zip(runs, faithfulness_results, relevancy_results, correctness_results):
    for metric_name, result in [("Faithfulness", f_result), ("Answer Relevancy", r_result), ("Answer Correctness", c_result)]:
        generation_rows.append({
            "설정": run["config"],
            "질문 ID": run["question_id"],
            "지표": metric_name,
            "점수": result.value,
        })
    print("평가:", run["config"], run["question_id"], "| Faithfulness:", round(f_result.value, 3), "| Answer Relevancy:", round(r_result.value, 3), "| Answer Correctness:", round(c_result.value, 3))

근거 이탈은 기준선과 같은 문서 4개를 넘기고 프롬프트만 바꾼 설정입니다. 이 설정의 Faithfulness가 기준선보다 낮다면 그 차이는 답변 생성 단계에서 생긴 것입니다.

### 6-2. 지표 6개 종합 표

지표 6개를 최종 결과, 검색, 생성의 세 묶음으로 나눠 읽습니다.

<img src="images/16_overall_reading.png" width="1050" alt="지표 6개를 읽는 순서. 최종 결과에서 문제를 찾고, 검색과 생성으로 원인을 좁힌다. 1 최종 결과: Answer Correctness, 답이 정답 답변과 맞는가. 2 검색: Context Recall, Context Precision, Context Relevancy, 필요한 문서를 위쪽에 가져왔는가. 3 생성: Faithfulness, Answer Relevancy, 검색 문서에 근거해 질문에 답했는가. 검색에서 문서가 빠지면 생성 지표도 함께 떨어지므로 검색부터 확인.">

- **최종 결과부터:** Answer Correctness로 답이 틀린 설정이 있는지 먼저 봅니다.
- **검색 다음 생성:** 검색 묶음이 괜찮을 때 생성 묶음을 봅니다. 필요한 문서가 빠지면 답변이 모자란 내용을 지어내기 쉬워 생성 지표도 함께 떨어지기 때문입니다.

In [ ]:
# 검색 지표 세 개와 생성 지표 세 개의 행을 한 목록으로 합칩니다. 행 하나가 설정·질문·지표 하나의 점수입니다
score_rows = precision_rows + recall_rows + relevance_rows + generation_rows

# 행은 설정, 열은 지표인 종합 표 (칸마다 질문 6개의 평균). 열은 최종 결과, 검색, 생성 순서로 둡니다
metric_order = ["Answer Correctness", "Context Recall", "Context Precision", "Context Relevancy", "Faithfulness", "Answer Relevancy"]
overall_table = pd.DataFrame(score_rows).pivot_table(index="설정", columns="지표", values="점수", aggfunc="mean").loc[CONFIGS, metric_order].round(2)  # 행은 기준선부터
display(overall_table)

평균은 질문 6개를 하나로 묶으므로, 한 질문의 큰 하락과 여러 질문의 작은 하락이 같은 값으로 보일 수 있습니다. 아래 표는 같은 점수를 질문별로 펼친 것입니다. 평균이 크게 움직인 지표가 있으면 어느 질문에서 움직였는지 이 표에서 찾습니다.

In [ ]:
# 설정·질문별로 지표 6개의 점수를 펼친 표 (행은 설정과 질문 ID, 열은 metric_order 순서)
per_question_table = pd.DataFrame(score_rows).pivot_table(index=["설정", "질문 ID"], columns="지표", values="점수").loc[CONFIGS, metric_order].round(2)
display(per_question_table)

### 6-3. 합격선으로 판정하기(절대 판정)

**합격선**은 지표마다 "이 점수 이상이면 서비스에 내보내도 된다"고 팀이 정한 값입니다.

- **도구의 기본값:** Ragas는 합격선을 정해 두지 않습니다. [DeepEval](https://deepeval.com/docs/metrics-introduction)은 지표마다 0.5, [Azure AI Foundry의 RAG 평가기](https://learn.microsoft.com/en-us/azure/ai-foundry/concepts/evaluation-evaluators/rag-evaluators)는 1~5점 가운데 3을 기본 합격선으로 둡니다. 둘 다 출발점일 뿐입니다.
- **실무에서 정하는 법:** 골든셋 답변 가운데 사람이 원문과 대조해 괜찮다고 본 답변들의 점수를 보고, 틀리면 피해가 큰 서비스일수록 높게 정합니다.
- **질문마다 셉니다:** 평균만 보면 한 질문의 큰 실패가 가려지므로, 질문마다 합격선을 넘었는지 세어 "합격한 질문 수"로 봅니다.

이 교안의 합격선은 점수 계산 방식에서 정한 예시값입니다.

| 지표 | 합격선 | 정한 근거 |
|---|---|---|
| Answer Correctness | 0.6 | 의미 유사도 몫(0.25 × 유사도)이 0.2 안팎이라, 정답 답변 주장의 절반 이상을 맞혀야 넘습니다 |
| Context Recall | 0.8 | 정답 답변 주장이 2~3개라 하나만 빠져도 0.5~0.67이 되어 미달입니다 |
| Context Precision | 0.7 | 문서 4개에서 1위 문서가 쓸모 있어야 넘고, 1위가 쓸모없으면 넘지 못합니다 |
| Context Relevancy | 0.75 | 점수가 0.25 단위라, 두 판정이 2점과 1점 이상이어야 넘습니다 |
| Faithfulness | 0.8 | 답변 주장 4개 가운데 하나만 근거가 없어도 0.75가 되어 미달입니다 |
| Answer Relevancy | 없음 | 교안 01 5-2처럼 엇나간 답변도 0 가까이 떨어지지 않아, 절댓값 대신 기준선과의 차이로 봅니다 |

In [ ]:
# 이 교안의 합격선. Answer Relevancy는 절댓값 기준을 두지 않습니다(교안 01 5-2)
THRESHOLDS = {
    "Answer Correctness": 0.6,   # 정답 답변 주장의 절반 이상을 맞힌 수준
    "Context Recall": 0.8,       # 정답 답변 주장 2~3개 가운데 하나라도 빠지면 미달
    "Context Precision": 0.7,    # 1위 문서가 쓸모 있어야 넘는 값
    "Context Relevancy": 0.75,   # 두 판정이 2점과 1점 이상이어야 넘는 값
    "Faithfulness": 0.8,         # 답변 주장 4개 가운데 하나만 근거가 없어도 미달
}

# 질문마다 합격선을 넘으면 1, 아니면 0 (NaN은 비교 결과가 False라 0으로 셉니다)
pass_rows = []
for row in score_rows:
    if row["지표"] in THRESHOLDS:
        pass_rows.append({"설정": row["설정"], "지표": row["지표"], "합격": int(row["점수"] >= THRESHOLDS[row["지표"]])})

# 설정마다 합격선을 넘은 질문 수 (질문 6개 중)
pass_table = pd.DataFrame(pass_rows).pivot_table(index="설정", columns="지표", values="합격", aggfunc="sum").loc[CONFIGS, list(THRESHOLDS)]
display(pass_table)

- **기준선 행부터 봅니다:** 기준선에서도 6개를 다 넘지 못한 지표가 있다면 기준선 자체가 부족한 것입니다. 기준선끼리는 차이가 0이라, 기준선 비교만으로는 이 부족함을 볼 수 없습니다.
- **고장 설정의 행:** 기준선보다 합격한 질문이 줄어든 지표가 이번 설정이 망가뜨린 곳입니다. 다음 절에서 그 위치를 기준선 비교로 좁힙니다.

### 6-4. 기준선과 비교해 고장 위치 찾기(상대 판정)

설정을 하나만 바꾸고 기준선과 비교하면, 떨어진 지표가 그 변경이 망가뜨린 단계를 가리킵니다. 여기서 설정 하나는 검색 문서 수, 문서 순서, 답변 프롬프트, 모델처럼 RAG를 이루는 요소 가운데 한 가지입니다. 이 교안의 `근거 이탈`은 프롬프트만 바꾼 설정입니다. 둘 이상을 한꺼번에 바꾸면 어느 변경이 지표를 떨어뜨렸는지 가를 수 없습니다. 지표마다 떨어지는 고장이 다르므로 다음 여섯 진단 이름으로 좁힐 수 있습니다.

| 진단 이름 | 뜻 | 이 교안에서 일부러 넣은 설정 |
|---|---|---|
| `검색 누락` | 필요한 문서를 가져오지 못함 | `검색 누락` |
| `순위 문제` | 쓸모 있는 문서를 가져왔지만 아래 순위에 있음 | `순위 흐트림` |
| `엉뚱한 검색` | 질문과 관련 없는 문서를 가져옴 | 없음(질문과 동떨어진 문서로 비교한 경우에 해당) |
| `생성 환각` | 검색 문서는 괜찮은데 답변이 문서에 없는 내용을 지어냄(교안 01의 할루시네이션 가운데 답변 생성 단계에서 생긴 것) | `근거 이탈` |
| `질문 이탈` | 문서에 근거해 답했지만 질문이 묻는 내용이 아님 | 없음(교안 01의 `질문과_다른_이야기`에 해당) |
| `답 불일치` | 검색과 근거는 괜찮은데 답이 정답 답변과 다름(문서가 낡았거나 답을 뽑지 못함) | 없음(교안 01 따라하기의 낡은 아동수당 문서에 해당) |

<img src="images/13_diagnosis.png" width="1050" alt="점수 패턴으로 고장 위치 찾기. 위에서부터 보고, 기준선보다 0.2 이상 떨어진 첫 지표로 진단한다. 1단계 검색 지표: Context Recall은 검색 누락(필요한 문서를 못 가져옴), Context Relevancy는 엉뚱한 검색(질문과 관련 없는 문서), Context Precision은 순위 문제(쓸모 있는 문서가 아래에 있음). 2단계 생성 지표, 검색 지표가 기준선과 비슷할 때: Faithfulness는 생성 환각(검색 문서에 없는 내용을 지어냄), Answer Relevancy는 질문 이탈(질문과 다른 내용을 답함), Answer Correctness는 답 불일치(문서가 낡았거나 답을 못 뽑음). 필요한 문서가 빠지면 생성 지표도 함께 떨어지므로 검색 지표부터 본다.">

**판정 순서**

- **떨어진 정도:** 지표마다 "기준선 값 - 이 설정 값"을 봅니다. 클수록 이 설정의 점수가 기준선보다 많이 내려간 것입니다.
- **위에서부터 하나만:** 표를 위에서부터 보고 처음 맞는 조건의 이름 하나만 붙입니다. 검색 지표를 먼저 보는 이유는, 필요한 문서가 빠지면 생성 지표도 함께 떨어지기 때문입니다. 생성 지표 가운데서는 받은 문서를 벗어났는지(Faithfulness), 질문에 맞는지(Answer Relevancy), 최종 답이 맞는지(Answer Correctness) 순서로 봅니다. Answer Correctness는 어느 단계가 고장 나도 함께 떨어지므로, 다른 지표가 모두 기준선과 비슷할 때만 이 지표로 이름을 붙입니다.

| 순서 | 조건 | 붙일 이름 |
|---|---|---|
| 1 | Context Recall이 기준선보다 0.2 이상 떨어짐 | `검색 누락` |
| 2 | Context Relevancy가 기준선보다 0.2 이상 떨어짐 | `엉뚱한 검색` |
| 3 | Context Precision이 기준선보다 0.2 이상 떨어짐 | `순위 문제` |
| 4 | Faithfulness가 기준선보다 0.2 이상 떨어짐 | `생성 환각` |
| 5 | Answer Relevancy가 기준선보다 0.2 이상 떨어짐 | `질문 이탈` |
| 6 | Answer Correctness가 기준선보다 0.2 이상 떨어짐 | `답 불일치` |
| 7 | 아무것도 해당하지 않음 | `기준선과 비슷` |

차이 0.2는 이 교안이 정한 값이며, 이 값을 권하는 공식 기준은 없습니다. 품질을 가르는 합격선이 아니라 "기준선과 뚜렷이 다르다"고 볼 차이이고, 질문 수나 자료가 바뀌면 알맞은 값도 달라집니다.

- **진단을 믿기 전에 질문 수를 셉니다:** 질문이 6개뿐이라 한 질문이 1에서 0으로 떨어지면 평균이 0.17 내려갑니다. 평가용 LLM의 판정이 한 질문에서만 흔들려도 평균 차이가 커질 수 있으므로, 6-2의 질문별 점수표에서 기준선보다 0.2 이상 떨어진 질문이 몇 개인지 봅니다. 한 질문뿐이면 그 질문의 원문부터 열어 봅니다.

**진단 이름이 정해지면 고칠 단계도 정해집니다**

- **검색 누락·엉뚱한 검색:** 검색 단계를 고칩니다. 검색 문서 수(k), 청크 나누기, 검색 방식을 바꾼 뒤 같은 질문으로 다시 평가합니다.
- **순위 문제:** 리랭킹으로 쓸모 있는 문서를 위로 올립니다.
- **생성 환각:** 답변 프롬프트를 고쳐 검색 문서 밖의 내용을 덧붙이지 않게 합니다.
- **질문 이탈:** 답변 프롬프트가 질문에 바로 답하게 하는지 확인합니다.
- **답 불일치:** 원문을 열어 문서가 낡았는지(검색 대상 문서를 최신으로 교체), 문서에 답이 있는데 못 뽑았는지(프롬프트·모델 확인) 가립니다.

### 6-5. 원문으로 확인

진단은 고장 위치를 가리킬 뿐이므로 원문을 열어 확인합니다. 아래 셀은 기준선의 질문마다 Context Recall, 정답 답변, 검색되지 않은 정답 문서의 본문을 출력합니다.

- 기준선에서도 Context Recall이 1보다 낮은 질문은 정답 문서가 상위 4개 밖에 있었을 수 있습니다.
- 그런 질문에 검색되지 않은 정답 문서가 있으면, 정답 답변의 주장이 그 문서에만 있는지 대조합니다. 그렇다면 손볼 곳은 답변 프롬프트가 아니라 검색입니다.

In [ ]:
# runs와 recall_rows는 같은 순서로 쌓였으므로 zip으로 짝지어 읽습니다
for run, recall_row in zip(runs, recall_rows):
    if run["config"] != "기준선":
        continue
    print("질문 ID:", run["question_id"], "| Context Recall:", round(recall_row["점수"], 2), "| 검색 문서:", ", ".join(run["retrieved_ids"]))
    print("정답 답변:", run["reference"])

    # 사람이 정한 정답 문서 가운데 검색되지 않은 문서의 본문 전체
    for doc_id in relevant_ids[run["question_id"]]:
        if doc_id not in run["retrieved_ids"]:
            print("검색되지 않은 정답 문서:", doc_id, documents_by_id[doc_id])
    print()

**확인 퀴즈**: 어떤 설정에서 Faithfulness는 기준선처럼 높은데 Context Recall만 크게 내려갔습니다. 답변 프롬프트에 "문서에 없는 내용은 말하지 마세요"를 더 강하게 넣으면 이 문제가 해결될까요?

<details><summary>정답 보기</summary>

아니요. 진단이 가리키는 곳은 검색입니다.

- Faithfulness가 높으므로 답변은 이미 검색 문서 안의 내용만으로 작성되고 있습니다.
- 필요한 문서가 검색 문서에 들어오지 않은 것이 문제라, 프롬프트를 바꿔도 빠진 문서는 들어오지 않습니다.

</details>

### 6-6. 실무에서 지표 고르기

실무에서는 지표 6종을 매번 모두 계산하지 않습니다. 무엇을 바꿨는지, 정답 답변(골든셋)이 있는지에 따라 몇 개만 고릅니다. Ragas 문서도 약한 신호를 주는 지표를 늘리지 말고 강한 신호를 주는 지표 몇 개를 고르라고 하고, DeepEval은 한 번에 5개 이하를 권합니다.

**상황별로 고르는 지표**

| 상황 | 쓸 지표 | 고른 이유 |
|---|---|---|
| 검색기 조정(청크 크기, 검색 문서 수 k, 임베딩 모델, 리랭커) | 정답 문서 ID가 있으면 앞 단원의 Recall@K·MRR, 없으면 Context Recall·Context Precision | 검색기만 바꿨으므로 생성 지표는 뺍니다. ID 기반 지표는 평가용 LLM을 부르지 않아 빠르고 실행마다 같은 값이 나옵니다 |
| 출시 전 프롬프트·모델 교체 | Faithfulness, Answer Relevancy, Answer Correctness를 같은 골든셋으로 기준선과 비교 | 생성 단계만 바꿨으므로 검색 지표는 이전 결과를 그대로 씁니다 |
| 운영 중 실제 질문 점검 | Faithfulness, Answer Relevancy, Context Relevancy를 들어온 질문 일부에만 | 실제 질문에는 정답 답변이 없어 Answer Correctness·Context Recall·Context Precision은 계산할 수 없습니다. 이 셋은 골든셋으로 정기 평가할 때 씁니다 |
| 틀리면 피해가 큰 분야(의료·법률) | 전문가가 쓴 정답 답변으로 Answer Correctness, 그리고 Faithfulness·Context Recall | Answer Relevancy만 보고 통과시키지 않습니다. 평가용 LLM의 판정을 전문가 판정과 대조해 맞춘 뒤 씁니다 |
| 예산이 빠듯할 때 | 검색은 ID 기반 지표, 생성은 질문 일부에만 Faithfulness | 아래 표처럼 지표마다 평가용 LLM 호출 수가 크게 다릅니다 |

**지표마다 드는 비용** (평가 샘플 하나에 평가용 LLM을 부르는 횟수, 설치된 ragas 0.4.3 기준)

| 지표 | 호출 수 | 정답 답변 |
|---|---|---|
| Faithfulness | 2 | 필요 없음 |
| Answer Relevancy | 3 (임베딩 모델도 부름) | 필요 없음 |
| Answer Correctness | 3 (임베딩 모델도 부름) | 필요 |
| Context Precision | 검색 문서 수만큼 | 필요 |
| Context Recall | 1 | 필요 |
| Context Relevancy | 2 | 필요 없음 |
| 앞 단원의 Recall@K·MRR | 0 | 정답 문서 ID 필요 |

예를 들어 질문 100개에 검색 문서 5개씩이면 여섯 지표를 모두 계산하는 데 평가용 LLM을 약 1,600번 부릅니다.

**지표를 읽을 때 주의할 점**

- **Answer Relevancy:** 사실이 맞는지는 보지 않습니다. 근거가 없어 "확인할 수 없습니다"라고 답한 올바른 거절도 회피성 답변으로 보아 0점을 줄 수 있습니다.
- **Faithfulness:** 검색 문서와 일치하는지만 봅니다. 낡거나 엉뚱한 문서를 충실히 옮긴 답도 1.0입니다.
- **Answer Correctness:** 점수의 25%가 의미 유사도라, 틀렸지만 표현이 비슷한 답도 부분 점수를 받습니다.
- **Context Relevancy:** 검색 문서 묶음 전체에 점수 하나를 0.25 단위로 매기므로, 어느 문서가 문제인지는 알려 주지 않습니다.
- **Context Precision:** 이름은 앞 단원의 Precision@K와 같지만, 순위마다 정밀도를 구해 평균하는 다른 계산입니다.
- **판정 흔들림:** 평가용 LLM의 판정은 실행마다 조금씩 다릅니다. 처음 쓸 때는 사람이 판정한 답변 수십 개와 대조해 평가용 LLM을 맞추고, 결정에 쓰는 비교는 두세 번 실행해 결과가 같은지 봅니다.

참고: [Ragas 지표 개요](https://docs.ragas.io/en/stable/concepts/metrics/overview/), [DeepEval RAG 평가 가이드](https://deepeval.com/guides/guides-rag-evaluation), [LangSmith RAG 평가](https://docs.langchain.com/langsmith/evaluate-rag-tutorial), [Azure AI Foundry RAG 평가기](https://learn.microsoft.com/en-us/azure/ai-foundry/concepts/evaluation-evaluators/rag-evaluators), [Hamel Husain, Evals FAQ](https://hamel.dev/blog/posts/evals-faq/)

## 7. 🖐️ 함께 따라하기: 출산·육아 지원금 검색 RAG 진단

본실습의 유연근무 매뉴얼과 다른 분야인 출산·육아 지원 서비스 문서에 같은 진단 방법을 적용합니다.

자료: 행정안전부 「대한민국 공공서비스(혜택) 정보」(보조금24, 공공데이터포털, 공공누리). 실제로 신청하기 전에는 소관 기관의 안내를 확인하세요.

**실습 진행 방법**

작성 셀은 이 함께 따라하기 절에만 있습니다.

- **부분 작성:** 셀에 남은 `...`만 주석 힌트에 맞춰 채웁니다. 나머지 코드는 제공합니다.
- `...`가 남은 채로 실행하면 에러가 나거나 빈 결과가 나옵니다. 작성 셀을 완성한 뒤 바로 아래 셀을 실행해 결과를 봅니다.
- 평가 셀이 네트워크 오류로 멈추면 그 셀만 다시 실행합니다. 결과 목록을 셀 안에서 빈 목록으로 새로 만들므로 점수가 중복으로 쌓이지 않습니다.
- 완성 코드는 수업이 끝난 뒤 강사가 공유합니다. 먼저 직접 작성해 봅니다.

### 7-1. 질문과 작성 목표

- **배경:** 지원금 상담 챗봇은 검색 문서 상위 5개를 답변 모델에 넘겨 왔습니다. 응답 비용을 줄이려고 상위 3개로 바꾼 뒤 "여러 지원을 함께 물으면 일부를 빠뜨린다"는 문의가 들어왔습니다. 바꾸기 전 설정을 `기준선`, 바꾼 설정을 `바꾼 뒤`로 두고 같은 질문 10개로 평가해, 바꾼 것이 어느 단계를 망가뜨렸는지 찾습니다.
- **질문 유형:** 질문 10개를 필요한 정답 문서 수로 나눕니다. 지원 서비스 두세 개를 함께 물어 정답 문서가 2~3개인 **복합 질문** 5개(`g1`~`g5`)와, 서비스 하나만 물어 정답 문서가 1개인 **단일 질문** 5개(`s1`~`s5`)입니다. 검색 문서 수를 바꾸면 영향을 받는 것이 정답 문서 수이므로 이 기준으로 나눕니다.
- **작성:** 작성 셀 네 개에 다음 네 가지를 차례로 작성합니다. 번호는 작성 셀의 주석 번호와 같습니다(2와 6은 제공 코드).
  1) `기준선`의 검색기 `baseline_retriever`(상위 5개)
  3) 실행마다 지표 6개(`metric_order` 순서)를 계산한 결과
  4) 설정 × 지표 평균 표 `practice_table`
  5) 질문마다 합격선을 넘었는지 표시한 값 `passed`
- **확인:** 합격선으로 두 설정을 써도 되는지 판정하고, 평균 표를 6-4의 판정 순서표로 읽어 `바꾼 뒤`의 진단 이름을 정합니다. 원인은 진단을 마친 뒤 열어 봅니다.
- **힌트:** 지표 객체, `metric_order`, `THRESHOLDS`는 3~6절에서 만든 것을 그대로 쓰고, `ascore`에 넘기는 인자도 3~6절의 평가 셀과 같습니다.

### 7-2. 지원 서비스 문서와 상담 프롬프트

In [ ]:
# 따라하기 문서가 미리 저장된 Chroma 컬렉션 열기 (본실습과 다른 컬렉션)
practice_vector_store = Chroma(
    collection_name="parenting_support",
    persist_directory=str(data_dir / "따라하기" / "chroma"),
    embedding_function=embedding_model,     # 본실습과 같은 임베딩 모델로 질문만 임베딩합니다
    create_collection_if_not_exists=False,  # 배포한 컬렉션이 없으면 새로 만들지 않고 오류를 냅니다
)
print("지원 서비스 문서:", len(practice_vector_store.get()["ids"]))

# 질문 10개와 정답 답변 읽기 (relevant_doc_ids는 사람이 정한 정답 문서)
practice_questions = json.loads((data_dir / "따라하기" / "questions.json").read_text(encoding="utf-8"))
practice_question_rows = []
for entry in practice_questions:
    practice_question_rows.append({
        "질문 ID": entry["question_id"],
        "유형": entry["question_type"],
        "질문": entry["question"],
        "정답 답변": entry["reference"],
        "정답 문서": ", ".join(entry["relevant_doc_ids"]),
    })
display(pd.DataFrame(practice_question_rows))

In [ ]:
# 본실습의 answer_prompt와 같은 모양이고 챗봇의 역할만 다릅니다
practice_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 출산·육아 지원금 상담 챗봇입니다. 아래 문서에 있는 내용만 근거로 2~4문장으로 답하세요. "
               "문서에 없는 내용은 추측하지 말고 문서에서 확인할 수 없다고 말하세요."),
    ("human", "문서:\n{context}\n\n질문: {question}"),
])
practice_chain = practice_prompt | llm

### 7-3. 두 설정 평가와 진단

질문 10개를 두 설정으로 실행하고, 실행마다 지표 6개를 `asyncio.gather`로 동시에 계산합니다. 본실습 6절처럼 평균 표, 질문별 표, 합격한 질문 수를 차례로 보고, 질문 유형별 평균으로 어느 유형에서 점수가 떨어졌는지 나눠 봅니다.

In [ ]:
# 1) 기준선: 바꾸기 전에 운영하던 설정처럼 상위 5개를 찾는 검색기를 만드세요 (본실습의 retriever와 같은 방법)
baseline_retriever = ...

# 바꾼 뒤: 답변 모델에 넘기는 문서를 줄이려고 상위 3개로 바꾼 검색기
changed_retriever = practice_vector_store.as_retriever(search_kwargs={"k": 3})

# 2) 기준선과 바꾼 뒤 두 설정으로 질문마다 답변 만들기
practice_runs = []
for entry in practice_questions:
    # 두 설정으로 각각 검색해 답변 입력을 만듭니다
    configs = ["기준선", "바꾼 뒤"]
    docs_list = [baseline_retriever.invoke(entry["question"]), changed_retriever.invoke(entry["question"])]
    inputs_list = []
    for docs in docs_list:
        inputs_list.append({"question": entry["question"], "context": format_documents(docs)})

    # 두 설정의 답변을 batch로 한 번에 받습니다 (결과는 입력 순서대로 돌아옵니다)
    answers = practice_chain.batch(inputs_list, config={"max_concurrency": MAX_CONCURRENCY})  # 본실습과 같은 동시 요청 상한

    for config, docs, answer in zip(configs, docs_list, answers):
        response = answer.text.strip()
        practice_runs.append({
            "config": config,
            "question_id": entry["question_id"],
            "user_input": entry["question"],
            "retrieved_contexts": [doc.page_content for doc in docs],
            "retrieved_ids": [evidence_id(doc) for doc in docs],
            "response": response,
            "reference": entry["reference"],
        })
        print("생성:", entry["question_id"], config, "| 검색 문서 수:", len(docs), "| 답변 앞부분:", response[:50])
print("실행 수:", len(practice_runs))

In [ ]:
# 3) 본실습에서 만든 지표 객체로 실행마다 지표 6개를 계산 (모든 실행을 한꺼번에 동시 실행)
async def score_practice_run(run):
    """실행 하나의 지표 6개를 metric_order 순서로 계산해 결과 목록을 돌려줍니다."""
    # asyncio.gather는 넘긴 순서대로 결과를 돌려줍니다. ascore마다 run_limited로 감싸 전체 동시 실행 수를 상한 안에 둡니다
    return await asyncio.gather(
        ...,  # run_limited(answer_correctness.ascore(...)): 질문·답변·정답 답변
        ...,  # run_limited(context_recall.ascore(...)): 질문·검색 문서·정답 답변
        ...,  # run_limited(context_precision.ascore(...)): 질문·정답 답변·검색 문서
        ...,  # run_limited(context_relevance.ascore(...)): 질문·검색 문서
        ...,  # run_limited(faithfulness.ascore(...)): 질문·답변·검색 문서
        ...,  # run_limited(answer_relevancy.ascore(...)): 질문·답변
    )

# 실행마다 끝나기를 기다리지 않고 모든 실행의 평가를 함께 시작합니다 (결과는 practice_runs 순서)
all_results = await tqdm_asyncio.gather(*[score_practice_run(run) for run in practice_runs], desc="따라하기 평가")

practice_rows = []
for run, results in zip(practice_runs, all_results):
    # 본실습과 같은 네 열로 지표마다 한 행씩 쌓기 (metric_order와 results는 같은 순서)
    for metric_name, result in zip(metric_order, results):
        practice_rows.append({
            "설정": run["config"],
            "질문 ID": run["question_id"],
            "지표": metric_name,
            "점수": result.value,
        })
    print("평가:", run["config"], run["question_id"], "| 점수:", [round(result.value, 2) for result in results])

In [ ]:
# 4) practice_rows로 행은 설정, 열은 지표(metric_order 순서)인 평균 표를 만드세요 (본실습의 overall_table과 같은 방법)
practice_table = ...
display(practice_table)

In [ ]:
# 설정·질문별로 지표 6개의 점수를 펼친 표 (본실습 6-2의 per_question_table과 같은 방법)
practice_per_question = pd.DataFrame(practice_rows).pivot_table(index=["설정", "질문 ID"], columns="지표", values="점수")[metric_order].round(2)
display(practice_per_question)

In [ ]:
# 5) 질문마다 합격선(본실습의 THRESHOLDS)을 넘었으면 1, 아니면 0으로 표시하세요
practice_pass_rows = []
for row in practice_rows:
    if row["지표"] in THRESHOLDS:
        passed = ...  # 점수가 그 지표의 합격선 이상이면 1, 아니면 0 (int로)
        practice_pass_rows.append({"설정": row["설정"], "지표": row["지표"], "합격": passed})

# 설정마다 합격선을 넘은 질문 수 (질문 10개 중)
practice_pass_table = pd.DataFrame(practice_pass_rows).pivot_table(index="설정", columns="지표", values="합격", aggfunc="sum")[list(THRESHOLDS)]
display(practice_pass_table)

In [ ]:
# 6) 질문 유형(복합·단일)별 평균 표. 전체 평균에 섞여 보이지 않던 유형별 차이를 나눠 봅니다
type_by_question = {}
for entry in practice_questions:
    type_by_question[entry["question_id"]] = entry["question_type"]

practice_df = pd.DataFrame(practice_rows)
practice_df["유형"] = practice_df["질문 ID"].map(type_by_question)  # 질문 ID로 유형 열을 붙입니다
practice_type_table = practice_df.pivot_table(index=["유형", "설정"], columns="지표", values="점수", aggfunc="mean")[metric_order].round(2)
display(practice_type_table)

**표를 보고 답해 볼 질문**

- 합격선 표에서 `바꾼 뒤`가 기준선보다 합격한 질문이 줄어든 지표는 무엇인가요? 기준선도 미달인 지표가 있나요?
- `바꾼 뒤`의 Context Recall과 Answer Correctness는 기준선보다 얼마나 내려갔나요?
- 질문별 표에서 `바꾼 뒤`의 Context Recall이 기준선보다 떨어진 질문은 몇 개이고, 어느 질문인가요? 평균 하락이 한두 질문에서만 온 것인가요?
- `바꾼 뒤`의 Faithfulness는 기준선과 비교해 어떤가요? 그 설정의 답변을 열어 보면 이유가 보이나요?
- 유형별 평균 표에서 `바꾼 뒤`의 Context Recall이 떨어진 것은 복합 질문인가요, 단일 질문인가요? 전체 평균 표만 봤을 때와 무엇이 다르게 보이나요?
- 평균 표를 6-4의 판정 순서표대로 위에서부터 읽으면, 기준선보다 0.2 이상 떨어진 첫 지표는 무엇이고 진단 이름은 무엇인가요? 그 지표 말고도 함께 떨어진 지표가 있다면 왜 함께 떨어졌는지 생각해 보세요.

**원인 확인**: 진단 이름을 정한 뒤 열어 보세요.

<details><summary>원인 보기</summary>

복합 질문 5개는 정답 문서 하나가 검색 4위나 5위에 있습니다. 상위 5개를 넘기던 기준선에는 이 문서가 들어가지만, `바꾼 뒤`는 상위 3개만 넘기므로 빠집니다. 그래서 정답 답변의 주장 가운데 그 문서에 있던 주장을 뒷받침할 문서가 없어 Context Recall이 내려갑니다. 단일 질문 5개는 정답 문서가 1위라 두 설정 모두 그 문서를 넘기므로 Context Recall이 그대로입니다. 전체 평균에서는 복합 질문의 하락이 단일 질문과 섞여 작게 보이고, 유형별로 나눠야 복합 질문에서만 떨어졌다는 것이 드러납니다.

- **함께 떨어진 지표:** Context Relevancy도 복합 질문에서 내려갑니다. 질문이 물은 지원 가운데 하나에 대한 문서가 빠져, 문서 묶음이 질문의 일부에만 답하기 때문입니다. 원인이 빠진 문서이므로 판정 순서표대로 Context Recall의 `검색 누락`으로 진단합니다.
- **오히려 오른 지표:** Context Precision은 `바꾼 뒤`가 더 높게 나올 수 있습니다. 쓸모없던 4·5위 문서가 빠져 넘긴 문서 가운데 쓸모 있는 문서의 비율이 올라가기 때문입니다. 정밀도 하나만 보면 바꾼 설정이 더 나아 보이는 이유입니다.
- **고칠 곳:** 답변 프롬프트가 아니라 검색기의 `k`입니다.
- **다음 할 일:** `k`를 다시 5로 돌리거나, 비용을 지키려면 검색 순위를 개선해(예: 리랭킹) 정답 문서를 상위 3개 안으로 올립니다. 고친 설정을 같은 질문으로 다시 평가해 Context Recall이 기준선 수준으로 돌아왔는지 확인합니다.

</details>

**생각해 보기**: `바꾼 뒤`의 답변 가운데 Faithfulness가 높게 나온 답변이 있습니다. 왜 그럴까요? 이 답변을 좋은 답변이라고 할 수 있을까요?

<details><summary>정답 보기</summary>

좋은 답변이라고 할 수 없습니다.

- **Faithfulness가 높은 이유:** 답변이 받은 문서 3개 안의 내용만 말하고 빠진 지원은 "문서에서 확인할 수 없다"고 했다면 근거 없는 주장이 없습니다.
- **좋은 답변이 아닌 이유:** 질문이 물은 지원 서비스 가운데 하나를 빠뜨렸습니다. 이 부족함은 Context Recall(필요한 문서가 빠짐)과 Answer Correctness(정답 답변의 주장을 덜 맞힘)가 함께 보여 줍니다. Faithfulness 하나만 보면 이 답변을 합격으로 잘못 판정합니다.

</details>

## 8. 핵심 정리

- **Context Precision:** 쓸모 있는 문서가 검색 순위 위쪽에 모였는지 평가합니다. 쓸모 있는 문서가 하나뿐이어도 1위면 만점이라, 빠진 문서는 점수에 나타나지 않습니다.
- **Context Recall:** 정답 답변의 주장 가운데 검색 문서로 뒷받침되는 주장의 비율입니다. 순위와 상관없이 필요한 정보가 빠졌는지 알려 줍니다.
- **Context Relevancy:** 정답 답변 없이 질문과 검색 문서 묶음의 관련성을 0·1·2점으로 두 번 매겨 평균합니다. 질문과 동떨어진 문서나 질문의 일부만 다루는 묶음은 점수가 내려가지만, 무엇이 빠졌는지는 알려 주지 않으므로 Context Recall로 확인합니다.
- **종합 평가:** 검색 지표 3개와 교안 01의 생성 지표 3개를 한 표로 모아, 최종 결과(Answer Correctness), 검색, 생성 순서로 읽습니다.
- **합격선(절대 판정):** 지표마다 팀이 정한 합격선으로 "써도 되는가"를 판정하고, 질문마다 합격 여부를 셉니다. Ragas에는 기본 합격선이 없고 도구의 기본값(DeepEval 0.5, Azure 3/5)은 출발점일 뿐이라, 사람이 판정한 답변으로 맞춥니다.
- **기준선 비교(상대 판정):** 기준선과 비교해 검색 지표를 먼저 보고, 검색 지표가 비슷할 때만 생성 지표(Faithfulness, Answer Relevancy, Answer Correctness)를 차례로 봅니다. 평균 차이가 한두 질문에서 왔는지 세고, 진단은 위치를 가리킬 뿐이므로 원문을 열어 확인합니다.
- **지표 고르기:** 바꾼 것과 골든셋 유무로 고릅니다. 검색기 조정은 ID 기반 지표나 Context Recall·Context Precision, 프롬프트·모델 교체는 생성 지표 셋, 운영 중 점검은 정답 답변이 필요 없는 Faithfulness·Answer Relevancy·Context Relevancy를 씁니다.